# Tutorial 04: PDPTW Problem Configurations

Learn how to configure PDPTW problems for different real-world scenarios and constraints.

**What you'll learn:**
- Configure PDPTW for different constraint levels
- Model scenarios with/without time windows, capacity, charging
- Understand trade-offs between problem complexity and solution quality
- Adapt PDPTW for various use cases (food delivery, logistics, etc.)

**Prerequisites:**
- Tutorial 01 (Quickstart)
- Tutorial 03 (Custom Problems)

**Time:** ~25 minutes

## 1. Setup and Imports

In [ ]:
# Standard imports
import numpy as np
import pandas as pd
import random

# VRP Toolkit
from vrp_toolkit.problems.pdptw import PDPTWInstance
from vrp_toolkit.algorithms.alns.solver import greedy_insertion_initial_solution

# Set seed for reproducibility
np.random.seed(42)
random.seed(42)

print("Imports successful! Ready to explore PDPTW configurations.")

## 2. Quick Start: Constraint Levels

PDPTW problems can have different **constraint levels** depending on your use case:

1. **Relaxed PDPTW**: Loose constraints (easy to solve)
2. **Standard PDPTW**: Typical constraints (moderate difficulty)
3. **Strict PDPTW**: Tight constraints (challenging)

Let's create the same problem with different constraint levels:

In [ ]:
# Helper function to create base order table
def create_base_order_table():
    return pd.DataFrame([
        # Depot
        {'ID': 0, 'Type': 'depot', 'X': 0.0, 'Y': 0.0, 'Demand': 0.0,
         'StartTime': 0.0, 'EndTime': 480.0, 'ServiceTime': 0.0, 'PartnerID': 0,
         'RealIndex': 0, 'RealType': 'depot'},
        # Order 1: Pickup and delivery
        {'ID': 1, 'Type': 'cp', 'X': 5.0, 'Y': 5.0, 'Demand': 10.0,
         'StartTime': 0.0, 'EndTime': 480.0, 'ServiceTime': 5.0, 'PartnerID': 2,
         'RealIndex': 1, 'RealType': 'cp'},
        {'ID': 2, 'Type': 'cd', 'X': 10.0, 'Y': 10.0, 'Demand': -10.0,
         'StartTime': 0.0, 'EndTime': 480.0, 'ServiceTime': 5.0, 'PartnerID': 1,
         'RealIndex': 2, 'RealType': 'cd'}
    ])

def create_distance_matrix(order_table):
    """Compute Euclidean distance matrix."""
    n = len(order_table)
    dist_matrix = np.zeros((n, n))
    coords = order_table[['X', 'Y']].values
    
    for i in range(n):
        for j in range(n):
            if i != j:
                dist_matrix[i, j] = np.linalg.norm(coords[i] - coords[j])
    return dist_matrix

# Base data
order_table = create_base_order_table()
distance_matrix = create_distance_matrix(order_table)
time_matrix = distance_matrix / 2.0  # Robot speed = 2.0

# Configuration 1: Relaxed (easy)
instance_relaxed = PDPTWInstance(
    order_table=order_table.copy(),
    distance_matrix=distance_matrix,
    time_matrix=time_matrix,
    robot_speed=2.0
)

# Configuration 2: Standard (moderate)
order_table_standard = order_table.copy()
order_table_standard.loc[order_table_standard['Type'] == 'cp', 'EndTime'] = 120.0
order_table_standard.loc[order_table_standard['Type'] == 'cd', 'EndTime'] = 180.0

instance_standard = PDPTWInstance(
    order_table=order_table_standard,
    distance_matrix=distance_matrix,
    time_matrix=time_matrix,
    robot_speed=2.0
)

# Configuration 3: Strict (challenging)
order_table_strict = order_table.copy()
order_table_strict.loc[order_table_strict['Type'] == 'cp', 'StartTime'] = 30.0
order_table_strict.loc[order_table_strict['Type'] == 'cp', 'EndTime'] = 60.0
order_table_strict.loc[order_table_strict['Type'] == 'cd', 'StartTime'] = 90.0
order_table_strict.loc[order_table_strict['Type'] == 'cd', 'EndTime'] = 120.0

instance_strict = PDPTWInstance(
    order_table=order_table_strict,
    distance_matrix=distance_matrix,
    time_matrix=time_matrix,
    robot_speed=2.0
)

print("Created 3 configurations:")
print(f"  Relaxed: Time windows (0-480 minutes)")
print(f"  Standard: Pickup (0-120), Delivery (0-180)")
print(f"  Strict: Pickup (30-60), Delivery (90-120)")

## 3. Understanding PDPTW Configuration Options

### 3.1 Time Window Configuration

**Purpose:** Control when orders can be served

**Key parameters:**
- `StartTime`: Earliest service time
- `EndTime`: Latest service time
- `ServiceTime`: Time required at node

**Trade-offs:**
- Tighter windows → More realistic but harder to solve
- Looser windows → Easier to solve but less realistic

In [ ]:
# Example scenarios
scenarios = {
    'Same-day delivery': {'pickup': (0, 240), 'delivery': (60, 300)},
    'Express delivery': {'pickup': (0, 60), 'delivery': (30, 90)},
    'Flexible delivery': {'pickup': (0, 480), 'delivery': (0, 480)},
    'Business hours': {'pickup': (540, 1020), 'delivery': (540, 1020)}  # 9am-5pm
}

print("Time window configurations for different scenarios:")
print(f"{'Scenario':<20} {'Pickup Window':<20} {'Delivery Window'}")
print("-" * 60)
for name, windows in scenarios.items():
    pickup_str = f"{windows['pickup'][0]}-{windows['pickup'][1]}"
    delivery_str = f"{windows['delivery'][0]}-{windows['delivery'][1]}"
    print(f"{name:<20} {pickup_str:<20} {delivery_str}")

### 3.2 Capacity Configuration

**Purpose:** Limit how much a vehicle can carry

**When to constrain:**
- Physical vehicles with weight/volume limits
- Delivery robots with small cargo space
- Multiple item types (food, packages, etc.)

**When to relax:**
- Single-item deliveries
- Unlimited capacity scenarios
- Initial planning without constraints

In [ ]:
# Create multi-order instance
order_table_multi = pd.DataFrame([
    # Depot
    {'ID': 0, 'Type': 'depot', 'X': 0, 'Y': 0, 'Demand': 0,
     'StartTime': 0, 'EndTime': 480, 'ServiceTime': 0, 'PartnerID': 0,
     'RealIndex': 0, 'RealType': 'depot'},
    # Order 1: Small (demand = 5)
    {'ID': 1, 'Type': 'cp', 'X': 3, 'Y': 4, 'Demand': 5,
     'StartTime': 0, 'EndTime': 480, 'ServiceTime': 5, 'PartnerID': 3,
     'RealIndex': 1, 'RealType': 'cp'},
    # Order 2: Large (demand = 15)
    {'ID': 2, 'Type': 'cp', 'X': 6, 'Y': 2, 'Demand': 15,
     'StartTime': 0, 'EndTime': 480, 'ServiceTime': 5, 'PartnerID': 4,
     'RealIndex': 2, 'RealType': 'cp'},
    # Deliveries
    {'ID': 3, 'Type': 'cd', 'X': 8, 'Y': 6, 'Demand': -5,
     'StartTime': 0, 'EndTime': 480, 'ServiceTime': 5, 'PartnerID': 1,
     'RealIndex': 3, 'RealType': 'cd'},
    {'ID': 4, 'Type': 'cd', 'X': 10, 'Y': 4, 'Demand': -15,
     'StartTime': 0, 'EndTime': 480, 'ServiceTime': 5, 'PartnerID': 2,
     'RealIndex': 4, 'RealType': 'cd'}
])

distance_matrix_multi = create_distance_matrix(order_table_multi)
time_matrix_multi = distance_matrix_multi / 2.0

# Test different capacities
capacities = [10, 20, 50]  # Total demand = 20

print("Capacity configuration impact:")
print(f"Total order demand: {order_table_multi[order_table_multi['Demand'] > 0]['Demand'].sum():.0f}\n")

for cap in capacities:
    print(f"\nCapacity = {cap}:")
    instance_cap = PDPTWInstance(
        order_table=order_table_multi,
        distance_matrix=distance_matrix_multi,
        time_matrix=time_matrix_multi,
        robot_speed=2.0
    )
    
    initial_sol = greedy_insertion_initial_solution(
        problem=instance_cap,
        num_vehicles=3,
        vehicle_capacity=cap,
        battery_capacity=200,
        battery_consume_rate=1,
        penalty_unvisit=1000,
        penalty_delay=50
    )
    
    print(f"  Can fit all orders: {cap >= 20}")
    print(f"  Routes needed: ~{2 if cap < 20 else 1}")

### 3.3 Battery/Range Configuration

**Purpose:** Model electric vehicles with limited range

**When relevant:**
- Electric delivery robots
- Electric vehicles (EVs)
- Drones with flight time limits

**When to ignore:**
- Gas-powered vehicles with ample range
- Short routes within battery capacity

In [ ]:
# Add charging station to order table
order_table_charging = order_table_multi.copy()
charging_station = pd.DataFrame([{
    'ID': 5, 'Type': 'charging', 'X': 5, 'Y': 5, 'Demand': 0,
    'StartTime': 0, 'EndTime': 480, 'ServiceTime': 10,  # 10 min to charge
    'PartnerID': 0, 'RealIndex': 5, 'RealType': 'charging'
}])

order_table_charging = pd.concat([order_table_charging, charging_station], ignore_index=True)

# Recompute matrices with charging station
distance_matrix_charging = create_distance_matrix(order_table_charging)
time_matrix_charging = distance_matrix_charging / 2.0

# Create instance with limited battery
instance_battery = PDPTWInstance(
    order_table=order_table_charging,
    distance_matrix=distance_matrix_charging,
    time_matrix=time_matrix_charging,
    robot_speed=2.0
)

print("Battery configuration:")
print(f"  Charging station added at node 5")
print(f"  Service time (charging): 10 minutes")
print(f"  Total nodes: {len(order_table_charging)}")
print(f"\nWith limited battery, routes may need to visit charging station")

## 4. Real-World Scenario Configurations

Let's configure PDPTW for different real-world use cases.

### 4.1 Food Delivery (Tight Time Windows)

In [ ]:
# Food delivery: Must deliver hot food quickly
food_delivery_table = pd.DataFrame([
    # Depot (central kitchen)
    {'ID': 0, 'Type': 'depot', 'X': 0, 'Y': 0, 'Demand': 0,
     'StartTime': 660, 'EndTime': 840, 'ServiceTime': 0, 'PartnerID': 0,  # 11am-2pm
     'RealIndex': 0, 'RealType': 'depot'},
    # Order 1: Lunch order
    {'ID': 1, 'Type': 'cp', 'X': 3, 'Y': 2, 'Demand': 2,
     'StartTime': 720, 'EndTime': 750, 'ServiceTime': 2, 'PartnerID': 2,  # Pick 12:00-12:30
     'RealIndex': 1, 'RealType': 'cp'},
    {'ID': 2, 'Type': 'cd', 'X': 8, 'Y': 5, 'Demand': -2,
     'StartTime': 730, 'EndTime': 780, 'ServiceTime': 2, 'PartnerID': 1,  # Deliver 12:10-1:00
     'RealIndex': 2, 'RealType': 'cd'}
])

dist_food = create_distance_matrix(food_delivery_table)
time_food = dist_food / 3.0  # Faster robot for food delivery

instance_food = PDPTWInstance(
    order_table=food_delivery_table,
    distance_matrix=dist_food,
    time_matrix=time_food,
    robot_speed=3.0
)

print("Food Delivery Configuration:")
print("  - Tight time windows (30 min pickup, 50 min delivery)")
print("  - Faster robot speed (3.0 vs 2.0)")
print("  - Short service times (2 min)")
print("  - Lunch time window (11am-2pm)")

### 4.2 Package Delivery (Flexible Time Windows)

In [ ]:
# Package delivery: All-day delivery windows
package_delivery_table = pd.DataFrame([
    # Depot (distribution center)
    {'ID': 0, 'Type': 'depot', 'X': 0, 'Y': 0, 'Demand': 0,
     'StartTime': 0, 'EndTime': 480, 'ServiceTime': 0, 'PartnerID': 0,  # 8-hour shift
     'RealIndex': 0, 'RealType': 'depot'},
    # Order 1: Package
    {'ID': 1, 'Type': 'cp', 'X': 5, 'Y': 3, 'Demand': 5,
     'StartTime': 0, 'EndTime': 240, 'ServiceTime': 5, 'PartnerID': 2,  # Morning pickup
     'RealIndex': 1, 'RealType': 'cp'},
    {'ID': 2, 'Type': 'cd', 'X': 12, 'Y': 8, 'Demand': -5,
     'StartTime': 0, 'EndTime': 480, 'ServiceTime': 5, 'PartnerID': 1,  # All-day delivery
     'RealIndex': 2, 'RealType': 'cd'}
])

dist_package = create_distance_matrix(package_delivery_table)
time_package = dist_package / 2.0

instance_package = PDPTWInstance(
    order_table=package_delivery_table,
    distance_matrix=dist_package,
    time_matrix=time_package,
    robot_speed=2.0
)

print("Package Delivery Configuration:")
print("  - Flexible time windows (240 min pickup, 480 min delivery)")
print("  - Standard robot speed (2.0)")
print("  - Moderate service times (5 min)")
print("  - Higher capacity needs (demand = 5 vs 2)")

## 5. Configuration Trade-offs

### 5.1 Feasibility vs Optimality

In [ ]:
# Compare relaxed vs strict configurations
configs = [
    ('Relaxed', instance_relaxed, 100),
    ('Standard', instance_standard, 50),
    ('Strict', instance_strict, 30)
]

print("Configuration Trade-offs:")
print(f"{'Config':<12} {'Capacity':<12} {'Battery':<12} {'Result'}")
print("-" * 60)

for name, inst, cap in configs:
    sol = greedy_insertion_initial_solution(
        problem=inst,
        num_vehicles=2,
        vehicle_capacity=cap,
        battery_capacity=200,
        battery_consume_rate=1,
        penalty_unvisit=1000,
        penalty_delay=50
    )
    
    status = "Feasible" if sol.is_feasible() else "Infeasible"
    print(f"{name:<12} {cap:<12} {200:<12} {status}")

print("\nObservations:")
print("  - Relaxed constraints → Always feasible, but may not be realistic")
print("  - Strict constraints → May be infeasible, need more vehicles/capacity")
print("  - Standard constraints → Balance between realism and solvability")

## 6. Best Practices for Configuration

**1. Start Simple, Add Complexity**
- Begin with relaxed constraints
- Gradually tighten until realistic
- Check feasibility at each step

**2. Match Constraints to Reality**
- Don't add constraints you don't need
- Food delivery: tight time windows ✓ capacity ✓ battery ?
- Package delivery: flexible windows ✓ capacity ✓ battery ?

**3. Test Sensitivity**
- How tight can time windows be?
- What's minimum vehicle capacity needed?
- When do you need charging stations?

**4. Document Your Assumptions**
- Why these time windows?
- What determines capacity?
- Where did speed estimates come from?

## 7. Summary

**What you learned:**
- ✅ Configure PDPTW for different constraint levels
- ✅ Adjust time windows, capacity, battery for realism
- ✅ Model real-world scenarios (food, package delivery)
- ✅ Understand feasibility vs optimality trade-offs

**Key configuration parameters:**
1. **Time Windows**: `StartTime`, `EndTime`, `ServiceTime`
2. **Capacity**: `Demand` values and solving parameters
3. **Battery**: `battery_capacity`, `battery_consume_rate`, charging stations
4. **Speed**: `robot_speed` affects travel times

**Configuration guidelines:**
- Relaxed → Easy to solve, less realistic
- Strict → Hard to solve, more realistic
- Standard → Balance for most use cases

**Next steps:**
- Try Tutorial 05 for sensitivity analysis on these configurations
- Try Tutorial 06 for custom algorithms
- Experiment with your own real-world constraints